# Band Director Agent Demo

## Setup

The setup process depends on whether the demo is in Colab or being run locally. Follow the instructions based on your setup.

### If Running in Colab

Add `LLM_API_KEY` and `DATABASE_URL` to your Colab Secrets before running.

### If Running Locally

Create a `.env` file in the same directory as `src/` with your credentials:

```.env
LLM_API_KEY=<YOUR_ANTHROPIC_CREDENTIALS>
DATABASE_URL=<YOUR_POSTGRES_CREDENTIALS>
```
---

> For a more in-depth guide on setting up credentials, check the [README](../README.md)

In [1]:
import os
import sys

# Load system and credentials from either Google Colab or .env
try:
    from google.colab import userdata
    # Clone the system into Colab
    !git clone https://github.com/chrismckie/Band-Class-Agent.git
    %cd Band-Class-Agent/demo

    os.environ["LLM_API_KEY"] = userdata.get("LLM_API_KEY")
    os.environ["DATABASE_URL"] = userdata.get("DATABASE_URL")
    print("Credentials loaded from Colab Secrets.")
except Exception:
    print("Running locally. Credentials will be loaded from .env")

%pip install -q anthropic psycopg2-binary python-dotenv

# Add src/ to path
REPO_SRC = os.path.normpath(os.path.join(os.getcwd(), "..", "src"))
sys.path.insert(0, REPO_SRC)
print(f"Source path: {REPO_SRC}")

Running locally. Credentials will be loaded from .env
Note: you may need to restart the kernel to use updated packages.
Source path: /Users/Chris/Developer/GitHub/school/CSCI-264/Band-Class-Agent/src


### Database Setup

Run this cell to create the database tables and populate them with data. If the database and data already exist, you can skip this cell.

In [4]:
import psycopg2
from pathlib import Path
from dotenv import load_dotenv

# If running locally, load the database credentials (Colab setup happens in previous cell)
if not os.environ.get("DATABASE_URL"):
    load_dotenv(Path("..") / ".env")

database_url = os.environ.get("DATABASE_URL")
if not database_url:
    raise ValueError("DATABASE_URL is not set. Add it to Colab Secrets or your .env file.")

schema_path = Path("..") / "src" / "database" / "schema.sql"
seed_path   = Path("..") / "src" / "database" / "seed_data.sql"

conn = psycopg2.connect(database_url)
try:
    with conn.cursor() as cur:
        cur.execute(schema_path.read_text())
    conn.commit()
    print("Tables created.")
except Exception as e:
    conn.rollback()
    print(f"Tables not created. Tables may already exist. ({e})")

try:
    with conn.cursor() as cur:
        cur.execute(seed_path.read_text())
    conn.commit()
    print("Data loaded.")
except Exception as e:
    conn.rollback()
    print(f"Data skipped. Data may already be present. ({e})")
finally:
    conn.close()

Tables not created. Tables may already exist. (relation "instrument_family" already exists
)
Data skipped. Data may already be present. (duplicate key value violates unique constraint "instrument_family_family_name_key"
DETAIL:  Key (family_name)=(Woodwinds) already exists.
)


In [3]:
from main import chat
from database.connection import get_connection

print("Import Successful.")

Import Successful.


The demo is now ready to run.

---


## Demo

### Single INSERT

Ask the agent to add one student to the roster. 
- Planner classifies the intent and extracts the student's name and grade
- Generator builds a parameterized `INSERT` 
- Validator checks the grade range
- Executor writes the row
- Formatter replies with natural language feedback.

In [5]:
response = chat("Add Emma Rodriguez, a 10th grader, to the student list.")
print(response)

[Planner] received: 'Add Emma Rodriguez, a 10th grader, to the student list.'
[Planner] plan: {'intent': 'INSERT', 'entity': 'students', 'is_batch': False, 'records': [{'first_name': 'Emma', 'last_name': 'Rodriguez', 'grade': 10}], 'filters': {}, 'updates': {}, 'requires_clarification': False, 'clarification_question': None}
[Generator] received plan: intent=INSERT, entity=students, is_batch=False
[Validator] is_valid=True, errors=[]
[Executor] received SQL: 'INSERT INTO students (first_name, last_name, grade) VALUES (%s, %s, %s)', is_batch=False
[Executor] success: rows affected: 1
Emma Rodriguez has been added to your student list as a 10th grader. You're all set!


#### Verify with a SELECT

Query the roster to confirm the row was saved.

In [6]:
response = chat("Which students are in grade 10?")
print(response)

[Planner] received: 'Which students are in grade 10?'
[Planner] plan: {'intent': 'SELECT', 'entity': 'students', 'is_batch': False, 'records': [], 'filters': {'grade': 10}, 'updates': {}, 'requires_clarification': False, 'clarification_question': None}
[Generator] received plan: intent=SELECT, entity=students, is_batch=False
[Generator] sql: 'SELECT s.student_id, s.first_name, s.last_name, s.grade FROM students s WHERE s.grade = %s', params: [10]
[Validator] is_valid=True, errors=[]
[Executor] received SQL: 'SELECT s.student_id, s.first_name, s.last_name, s.grade FROM students s WHERE s.grade = %s', is_batch=False
[Executor] success: 8 row(s) returned
There are **8 students in grade 10**:

1. Sophia Garcia
2. Mason Taylor
3. Isabella Anderson
4. Logan Thomas
5. Mia Jackson
6. Lucas White
7. Emma Rodriguez
8. Emma Rodriguez

⚠️ **Heads up:** It looks like **Emma Rodriguez** appears twice in the list. This may be a duplicate entry worth checking out!


---

### Batch INSERT

When a request contains multiple records:
- Planner sets `is_batch: true`
- Validador checks all records before any are inserted
- Executor uses a single transaction
- If one record fails, none are committed.

In [7]:
response = chat(
    "Add these students: "
    "Jake Alvarez grade 11, Maria Chen grade 9, Devon Hill grade 12."
)
print(response)

[Planner] received: 'Add these students: Jake Alvarez grade 11, Maria Chen grade 9, Devon Hill grade 12.'
[Planner] plan: {'intent': 'INSERT', 'entity': 'students', 'is_batch': True, 'records': [{'first_name': 'Jake', 'last_name': 'Alvarez', 'grade': 11}, {'first_name': 'Maria', 'last_name': 'Chen', 'grade': 9}, {'first_name': 'Devon', 'last_name': 'Hill', 'grade': 12}], 'filters': {}, 'updates': {}, 'requires_clarification': False, 'clarification_question': None}
[Generator] received plan: intent=INSERT, entity=students, is_batch=True
[Validator] is_valid=True, errors=[]
[Executor] received SQL: 'INSERT INTO students (first_name, last_name, grade) VALUES (%s, %s, %s)', is_batch=True
[Executor] success: rows affected: 3
3 new students have been added to your roster: Jake Alvarez (11th grade), Maria Chen (9th grade), and Devon Hill (12th grade). They're all set and ready to be assigned instruments, music, or whatever comes next!


#### Query all students to confirm

A no-filter SELECT returns the full roster, showing all four students now in the system.

In [8]:
response = chat("Show me all students.")
print(response)

[Planner] received: 'Show me all students.'
[Planner] plan: {'intent': 'SELECT', 'entity': 'students', 'is_batch': False, 'records': [], 'filters': {}, 'updates': {}, 'requires_clarification': False, 'clarification_question': None}
[Generator] received plan: intent=SELECT, entity=students, is_batch=False
[Generator] sql: 'SELECT student_id, first_name, last_name, grade FROM students', params: []
[Validator] is_valid=True, errors=[]
[Executor] received SQL: 'SELECT student_id, first_name, last_name, grade FROM students', is_batch=False
[Executor] success: 33 row(s) returned
Here's a summary of all **33 students** in your band program, broken down by grade:

**9th Grade (8 students)**
Emma Johnson, Liam Smith, Olivia Davis, Noah Wilson, Ava Martinez, Ethan Brown, Elizabeth King, Maria Chen (×2)

**10th Grade (7 students)**
Sophia Garcia, Mason Taylor, Isabella Anderson, Logan Thomas, Mia Jackson, Lucas White, Emma Rodriguez (×2)

**11th Grade (7 students)**
Charlotte Harris, James Martin

---

### Query on a Different Table

The Planner identifies the target `music` as the target table.

In [9]:
response = chat("Add the piece Commando March by Karl King, difficulty 3.")
print(response)

[Planner] received: 'Add the piece Commando March by Karl King, difficulty 3.'
[Planner] plan: {'intent': 'INSERT', 'entity': 'music', 'is_batch': False, 'records': [{'title': 'Commando March', 'composer': 'Karl King', 'difficulty': 3}], 'filters': {}, 'updates': {}, 'requires_clarification': False, 'clarification_question': None}
[Generator] received plan: intent=INSERT, entity=music, is_batch=False
[Validator] is_valid=True, errors=[]
[Executor] received SQL: 'INSERT INTO music (title, composer, difficulty) VALUES (%s, %s, %s)', is_batch=False
[Executor] success: rows affected: 1
You've successfully added **Commando March** by Karl King to your music inventory, listed at a difficulty level of 3.


In [10]:
response = chat("Show me all pieces with difficulty 3.")
print(response)

[Planner] received: 'Show me all pieces with difficulty 3.'
[Planner] plan: {'intent': 'SELECT', 'entity': 'music', 'is_batch': False, 'records': [], 'filters': {'difficulty': 3}, 'updates': {}, 'requires_clarification': False, 'clarification_question': None}
[Generator] received plan: intent=SELECT, entity=music, is_batch=False
[Generator] sql: 'SELECT m.music_id, m.title, m.composer, m.difficulty FROM music m WHERE m.difficulty = %s', params: [3]
[Validator] is_valid=True, errors=[]
[Executor] received SQL: 'SELECT m.music_id, m.title, m.composer, m.difficulty FROM music m WHERE m.difficulty = %s', is_batch=False
[Executor] success: 2 row(s) returned
You have **2 pieces** at difficulty level 3:

1. **Commando March** – Karl King
2. **Commando March** – Karl King

Interestingly, both entries have the same title and composer. These might be duplicates in your inventory, or possibly two separate copies/editions of the same piece. You may want to double-check your records to see if one o

---

### Validation

The Validator enforces business rules before executing any SQL
- Only grades 9–12 are valid (high school only)
- The request below includes grade 8, and is rejected with a friendly explanation
- No data is written to the database.

In [12]:
response = chat("Add Alex Turner, an 8th grader, to the student list.")
print(response)

[Planner] received: 'Add Alex Turner, an 8th grader, to the student list.'
[Planner] plan: {'intent': 'INSERT', 'entity': 'students', 'is_batch': False, 'records': [{'first_name': 'Alex', 'last_name': 'Turner', 'grade': 8}], 'filters': {}, 'updates': {}, 'requires_clarification': False, 'clarification_question': None}
[Generator] received plan: intent=INSERT, entity=students, is_batch=False
[Validator] is_valid=False, errors=['Record: grade must be an integer between 9 and 12 (got 8).']
It looks like Alex Turner couldn't be added because the system only tracks students in grades 9–12 (high school). If Alex is an 8th grader joining the high school band, you may want to double-check the grade and re-enter it as 9, 10, 11, or 12.


---

## Cleanup

Remove all records inserted during this demo.

In [13]:
conn = get_connection()
try:
    with conn.cursor() as cur:
        cur.execute(
            "DELETE FROM students WHERE last_name IN (%s, %s, %s, %s)",
            ("Rodriguez", "Alvarez", "Chen", "Hill"),
        )
        cur.execute("DELETE FROM music WHERE title = %s", ("Commando March",))
    conn.commit()
    print("Demo data removed.")
except Exception as e:
    conn.rollback()
    print(f"Cleanup failed: {e}")
finally:
    conn.close()

Demo data removed.
